<a href="https://colab.research.google.com/github/SajjadRahati1/Index/blob/main/Use_FiassIntoAdvNGCF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 27.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
from collections import defaultdict


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


###پارامترها

In [ ]:
K = 10
retrieval_k = 200  # باید خیلی بزرگ‌تر از K باشه
ef_construction = 80
ef_search = 128

###بارگذاری Embedding‌ها

In [ ]:
path = '/content/drive/MyDrive/Dataset/movie-lens/'
# --- 1. خواندن embeddingها از فایل ---
user_embeddings = np.loadtxt(path + 'output/final_user_embeddings.txt', dtype=np.float32)
item_embeddings = np.loadtxt(path + 'output/final_item_embeddings.txt', dtype=np.float32)

In [ ]:
# نرمال‌سازی L2 برای هر بردار
faiss.normalize_L2(user_embeddings)
faiss.normalize_L2(item_embeddings)

###ساخت FAISS Index

In [ ]:
# --- 3. ساخت ایندکس FAISS با HNSW ---
dimension = item_embeddings.shape[1]
#حالت عادی :
index = faiss.IndexHNSWFlat(dimension, 32)

index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search

In [ ]:
# --- 4. افزودن بردار آیتم‌ها به ایندکس ---
index.add(item_embeddings)

###بارگذاری train و test dict

In [ ]:
#test and train

def load_user_item_dict(filepath):
    user_item_dict = defaultdict(set)
    with open(filepath, 'r') as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue
            user = int(tokens[0])
            items = list(map(int, tokens[1:]))
            user_item_dict[user] = set(items)
    return user_item_dict

train_dict = load_user_item_dict(path + 'train.txt')
test_dict = load_user_item_dict(path + 'test.txt')


###اجرای FAISS برای کاربران (تست) و جمع‌آوری پیشنهادها

In [ ]:
# faiss_preds = {}  # user_id -> top-K recommended item ids

# for user_id in test_dict.keys():
#     query = user_embeddings[user_id].reshape(1, -1)
#     _, indices = index.search(query, K)
#     faiss_preds[user_id] = set(indices[0])


###تعریف معیارهای ارزیابی

In [ ]:
def precision_at_k(recommended, ground_truth, k):
    return len(set(recommended[:k]) & set(ground_truth)) / k

def recall_at_k(recommended, ground_truth, k):
    return len(set(recommended[:k]) & set(ground_truth)) / len(ground_truth) if ground_truth else 0

def hit_rate_at_k(recommended, ground_truth, k):
    return 1.0 if len(set(recommended[:k]) & set(ground_truth)) > 0 else 0

def ndcg_at_k(recommended, ground_truth, k):
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in ground_truth:
            dcg += 1 / np.log2(i + 2)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(ground_truth), k)))
    return dcg / idcg if idcg > 0 else 0

In [ ]:

def evaluate_with_faiss(
    user_embeddings: np.ndarray,
    item_embeddings: np.ndarray,
    train_dict: dict,
    test_dict: dict,
    K: int = 10,
    retrieval_k: int = 200,
    verbose: bool = True
) -> dict:
    """
    ارزیابی embeddingها با FAISS و حذف آیتم‌های دیده‌شده در train
    """

    #ارزیابی کاربران
    precisions, recalls, ndcgs, hits = [], [], [], []
    valid_users = [u for u in test_dict if u in train_dict and len(test_dict[u]) > 0]

    for user_id in valid_users:
        query = user_embeddings[user_id].reshape(1, -1)
        _, raw_indices = index.search(query, retrieval_k)

        # حذف آیتم‌های دیده‌شده
        pred = [item for item in raw_indices[0] if item not in train_dict[user_id]]
        if len(pred) < K:
            continue

        gt = test_dict[user_id]
        precisions.append(precision_at_k(pred, gt, K))
        recalls.append(recall_at_k(pred, gt, K))
        ndcgs.append(ndcg_at_k(pred, gt, K))
        hits.append(hit_rate_at_k(pred, gt, K))
    #نتایج نهایی
    if verbose:
        print(f"Total evaluated users: {len(precisions)}/{len(valid_users)}")
        if len(precisions) == 0:
            print("No users had enough valid predictions for evaluation.")
        else:
            print(f"Precision@{K}: {np.mean(precisions):.4f}")
            print(f"Recall@{K}: {np.mean(recalls):.4f}")
            print(f"NDCG@{K}: {np.mean(ndcgs):.4f}")
            print(f"HitRate@{K}: {np.mean(hits):.4f}")

    return {
        "precision": np.mean(precisions) if precisions else 0.0,
        "recall": np.mean(recalls) if recalls else 0.0,
        "ndcg": np.mean(ndcgs) if ndcgs else 0.0,
        "hit_rate": np.mean(hits) if hits else 0.0,
        "evaluated_users": len(precisions)
    }


In [ ]:
results = evaluate_with_faiss(
    user_embeddings=user_embeddings,
    item_embeddings=item_embeddings,
    train_dict=train_dict,
    test_dict=test_dict,
    K=10,
    retrieval_k=200,
    verbose=True
)

Total evaluated users: 459/459
Precision@10: 0.2569
Recall@10: 0.0849
NDCG@10: 0.2742
HitRate@10: 0.7625


حالت 3 :
Total evaluated users: 458/459
Precision@10: 0.2262
Recall@10: 0.0728
NDCG@10: 0.2352
HitRate@10: 0.7358

حالت 3 با نرمالایز :
Total evaluated users: 458/459
Precision@10: 0.2382
Recall@10: 0.0755
NDCG@10: 0.2536
HitRate@10: 0.7467

حالت 2 با نرمالایز :
Total evaluated users: 458/459
Precision@10: 0.2454
Recall@10: 0.0769
NDCG@10: 0.2607
HitRate@10: 0.7424

In [ ]:
# ذخیره فایل
# with open(file_path, "w") as f:
#     f.write(code)

#تست حالات دیگه

In [ ]:

#حالت استفاده از ضرب داخلی :
index = faiss.IndexHNSWFlat(dimension, 32, faiss.METRIC_INNER_PRODUCT)
# index = faiss.IndexFlatIP(dimension)

index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search
index.add(item_embeddings)
results = evaluate_with_faiss(
    user_embeddings=user_embeddings,
    item_embeddings=item_embeddings,
    train_dict=train_dict,
    test_dict=test_dict,
    K=10,
    retrieval_k=200,
    verbose=True
)

Total evaluated users: 459/459
Precision@10: 0.2569
Recall@10: 0.0849
NDCG@10: 0.2742
HitRate@10: 0.7625


In [ ]:
index = faiss.IndexFlatIP(dimension)

index.add(item_embeddings)
results = evaluate_with_faiss(
    user_embeddings=user_embeddings,
    item_embeddings=item_embeddings,
    train_dict=train_dict,
    test_dict=test_dict,
    K=10,
    retrieval_k=200,
    verbose=True
)

Total evaluated users: 459/459
Precision@10: 0.2569
Recall@10: 0.0849
NDCG@10: 0.2742
HitRate@10: 0.7625


## ساخت کلاس MyHnsw برای بررسی با ضرب داخلی

In [ ]:
import numpy as np
import heapq

class MyHNSW:
    def __init__(self, dimension, M=32, ef_construction=400, ef_search=500):
        """
        HNSW Class for nearest neighbor search using Inner Product (Dot Product).
        :param dimension: the dimension of the embeddings.
        :param M: the number of neighbors for each node.
        :param ef_construction: the number of nearest neighbors used during construction.
        :param ef_search: the number of nearest neighbors used during search.
        """
        self.dimension = dimension
        self.M = M  # max number of neighbors per node
        self.ef_construction = ef_construction
        self.ef_search = ef_search
        self.data = []
        self.graph = {}
        self.entry_point = None

    def _add_node(self, node_id, embedding):
        """
        Add a node to the graph with its embedding.
        :param node_id: the unique id of the node (e.g., user or item).
        :param embedding: the vector representation of the node.
        """
        self.data.append((node_id, embedding))
        self.graph[node_id] = []

    def _search_neighbors(self, query_embedding, ef):
        """
        Perform a search to find the nearest neighbors using dot product similarity.
        :param query_embedding: the query vector.
        :param ef: the number of neighbors to consider during search.
        :return: list of neighbors sorted by similarity.
        """
        heap = []
        distances = []
        indices = []
        for node_id, embedding in self.data:
            similarity = np.dot(query_embedding, embedding)  # inner product (dot product)
            heapq.heappush(heap, (-similarity, node_id))  # we use negative similarity to have max-heap

        for _ in range(min(ef, len(heap))):
            dist, node = heapq.heappop(heap)
            distances.append(-dist)  # revert the negative distance to positive
            indices.append(node)

        return distances, indices

    def add(self, embeddings):
        """
        Adds a batch of embeddings to the graph.
        :param embeddings: list of tuples (node_id, embedding).
        """
        for node_id, embedding in embeddings:
            self._add_node(node_id, embedding)

    def search(self, query_embedding, k):
        """
        Search for k nearest neighbors for a given query using dot product similarity.
        :param query_embedding: the query embedding vector.
        :param k: number of nearest neighbors to return.
        :return: tuple (distances, indices) of top k nearest neighbors.
        """
        distances, indices = self._search_neighbors(query_embedding, self.ef_search)
        return distances[:k], indices[:k]  # return only the top-k neighbors


In [ ]:
def evaluate_with_hnsw(index, user_embeddings, item_embeddings, train_dict, test_dict, K=10, retrieval_k=200):
    """
    Evaluate the model using the HNSW index with precision, recall, NDCG, and HitRate.
    :param index: the HNSW index for nearest neighbor search.
    :param user_embeddings: embeddings for users.
    :param item_embeddings: embeddings for items.
    :param train_dict: train data in the form of {user_id: set of item_ids}.
    :param test_dict: test data in the form of {user_id: set of item_ids}.
    :param K: the number of top recommendations to consider.
    :param retrieval_k: the number of neighbors to retrieve from HNSW.
    :return: evaluation metrics.
    """
    precisions, recalls, ndcgs, hits = [], [], [], []
    valid_users = [u for u in test_dict if u in train_dict and len(test_dict[u]) > 0]

    for user_id in valid_users:
        query = user_embeddings[user_id].reshape(1, -1)
        distances, raw_indices = index.search(query, retrieval_k)

        # حذف آیتم‌های دیده‌شده
        pred = [item for item in raw_indices if item not in train_dict[user_id]]
        if len(pred) < K:
            continue

        gt = test_dict[user_id]
        precisions.append(precision_at_k(pred, gt, K))
        recalls.append(recall_at_k(pred, gt, K))
        ndcgs.append(ndcg_at_k(pred, gt, K))
        hits.append(hit_rate_at_k(pred, gt, K))

    # محاسبه میانگین مقادیر
    precision = np.mean(precisions) if precisions else 0.0
    recall = np.mean(recalls) if recalls else 0.0
    ndcg = np.mean(ndcgs) if ndcgs else 0.0
    hit_rate = np.mean(hits) if hits else 0.0

    print(f"Total evaluated users: {len(precisions)}/{len(valid_users)}")
    print(f"Precision@{K}: {precision:.4f}")
    print(f"Recall@{K}: {recall:.4f}")
    print(f"NDCG@{K}: {ndcg:.4f}")
    print(f"HitRate@{K}: {hit_rate:.4f}")

    return {
        "precision": precision,
        "recall": recall,
        "ndcg": ndcg,
        "hit_rate": hit_rate,
        "evaluated_users": len(precisions)
    }


In [ ]:
# ساخت ایندکس HNSW
dimension = item_embeddings.shape[1]
hnsw_index = MyHNSW(dimension=64)
hnsw_index.add([(i, item_embeddings[i]) for i in range(len(item_embeddings))])

# دیکشنری‌های train و test (فرض کنید این‌ها به درستی بارگذاری شده‌اند)
train_dict = load_user_item_dict(path + 'train.txt')
test_dict = load_user_item_dict(path + 'test.txt')

# ارزیابی مدل
results = evaluate_with_hnsw(
    index=hnsw_index,
    user_embeddings=user_embeddings,
    item_embeddings=item_embeddings,
    train_dict=train_dict,
    test_dict=test_dict,
    K=10,
    retrieval_k=200
)
print("Evaluation Results: ", results)

Total evaluated users: 459/459
Precision@10: 0.2569
Recall@10: 0.0849
NDCG@10: 0.2742
HitRate@10: 0.7625
Evaluation Results:  {'precision': np.float64(0.25686274509803925), 'recall': np.float64(0.08487475157862204), 'ndcg': np.float64(0.27420034564270884), 'hit_rate': np.float64(0.7625272331154684), 'evaluated_users': 459}


#تست با نمونه تستی

In [ ]:
import numpy as np

# پارامترها
n_users      = 100          # تعداد کاربران ساختگی
n_items      = 1000         # تعداد آیتم‌ها
latent_dim   = 64           # ابعاد نهفته
train_items  = 20           # آیتم‌های دیده‌شده در train برای هر کاربر
test_items   = 10           # آیتم‌های دیده‌نشده در test برای هر کاربر

rng = np.random.default_rng(42)

# ۱) embedding حقیقیِ «ایده‌آل»
true_user_emb  = rng.standard_normal((n_users, latent_dim)).astype(np.float32)
true_item_emb  = rng.standard_normal((n_items, latent_dim)).astype(np.float32)

# ۲) دادهٔ ضمنی (train/test) بر اساس بالاترین شباهتِ ضرب‌داخلی
scores = true_user_emb @ true_item_emb.T           # (n_users × n_items)

train_dict, test_dict = {}, {}
for u in range(n_users):
    ranked = scores[u].argsort()[::-1]             # بزرگ‌تر = ‌مشابه‌تر
    train_dict[u] = set(ranked[:train_items])
    test_dict[u]  = set(ranked[train_items:train_items+test_items])


In [ ]:
# ۲. embedding‌ های نویزی (شبیه خروجی مدل)
noise_scale = 0.00
user_emb = (true_user_emb + noise_scale *
            rng.standard_normal(true_user_emb.shape)).astype(np.float32, copy=False)
item_emb = (true_item_emb + noise_scale *
            rng.standard_normal(true_item_emb.shape)).astype(np.float32, copy=False)


import faiss, numpy as np
faiss.normalize_L2(user_emb)
faiss.normalize_L2(item_emb)


In [ ]:
index = faiss.IndexFlatIP(dimension)
index.add(item_emb)

metrics = evaluate_with_faiss(
    user_embeddings=user_emb,
    item_embeddings=item_emb,
    train_dict=train_dict,
    test_dict=test_dict,
    K=10,
    retrieval_k=40,
    verbose=True
)


Total evaluated users: 100/100
Precision@10: 0.6190
Recall@10: 0.6190
NDCG@10: 0.6708
HitRate@10: 1.0000


In [ ]:
user_id = 0
query = user_emb[user_id].reshape(1, -1)
_, raw_indices = index.search(query, 40)

print("Raw indices (retrieved):", raw_indices[0])
print("Train items:", sorted(train_dict[user_id]))
print("Intersection (should be many):", set(raw_indices[0]) & train_dict[user_id])


Raw indices (retrieved): [409 861 853 343 458 634 771 981 460 174 972 393 152 425 617 143 696 515
 121 860 172 298 613 184 288 580 791 156 435 987 893 884 739 899  21 966
 588 689 807 826]
Train items: [np.int64(152), np.int64(174), np.int64(298), np.int64(343), np.int64(393), np.int64(409), np.int64(425), np.int64(435), np.int64(458), np.int64(460), np.int64(515), np.int64(617), np.int64(634), np.int64(696), np.int64(771), np.int64(853), np.int64(860), np.int64(861), np.int64(972), np.int64(981)]
Intersection (should be many): {np.int64(515), np.int64(771), np.int64(393), np.int64(152), np.int64(409), np.int64(425), np.int64(298), np.int64(174), np.int64(435), np.int64(696), np.int64(458), np.int64(460), np.int64(972), np.int64(853), np.int64(981), np.int64(343), np.int64(860), np.int64(861), np.int64(617), np.int64(634)}


In [ ]:
dimension = latent_dim
index = faiss.IndexHNSWFlat(dimension, 32)
index.hnsw.efConstruction = 80
index.hnsw.efSearch       = 128
index.add(item_emb)


metrics = evaluate_with_faiss(
    user_embeddings = user_emb,
    item_embeddings = item_emb,
    train_dict      = train_dict,
    test_dict       = test_dict,
    K               = 10,
    retrieval_k     = 200,
    verbose         = True
)
print(metrics)


Total evaluated users: 100/100
Precision@10: 0.6190
Recall@10: 0.6190
NDCG@10: 0.6708
HitRate@10: 1.0000
{'precision': np.float64(0.6190000000000001), 'recall': np.float64(0.6190000000000001), 'ndcg': np.float64(0.6707778222841024), 'hit_rate': np.float64(1.0), 'evaluated_users': 100}


In [ ]:
user_id = 0
scores = true_user_emb @ true_item_emb.T
ranked = scores[user_id].argsort()[::-1]

print("Top 30 items:")
print(ranked[:30])
print("Train items:", train_dict[user_id])
print("Test items:", test_dict[user_id])


Top 30 items:
[409 343 861 853 634 458 174 460 771 515 298 617 152 981 425 393 435 696
 860 972 613 121 143 826 288 172 314 987 358 970]
Train items: {np.int64(771), np.int64(515), np.int64(393), np.int64(152), np.int64(409), np.int64(425), np.int64(298), np.int64(174), np.int64(435), np.int64(696), np.int64(458), np.int64(460), np.int64(972), np.int64(853), np.int64(981), np.int64(343), np.int64(860), np.int64(861), np.int64(617), np.int64(634)}
Test items: {np.int64(288), np.int64(613), np.int64(358), np.int64(970), np.int64(172), np.int64(314), np.int64(143), np.int64(121), np.int64(826), np.int64(987)}


In [ ]:
# بعد از ساخت ایندکس
D, I = index.search(user_emb[0:1], 10)
print("Top-10 retrieved item indexes for user 0:", I[0])
print("Ground-truth test items:", test_dict[0])


Top-10 retrieved item indexes for user 0: [409 861 853 343 458 634 771 981 460 174]
Ground-truth test items: {np.int64(288), np.int64(613), np.int64(358), np.int64(970), np.int64(172), np.int64(314), np.int64(143), np.int64(121), np.int64(826), np.int64(987)}


In [ ]:
print(index.metric_type)


1


In [ ]:
print("Total items:", len(item_emb))
print("Total items in FAISS index:", index.ntotal)


Total items: 1000
Total items in FAISS index: 1000


In [ ]:
import numpy as np
import faiss

# پارامترها
n_users     = 100
n_items     = 1000
latent_dim  = 64
train_items = 20
test_items  = 10
noise_scale = 0.0

rng = np.random.default_rng(42)

# داده ایده‌آل (بدون نویز)
true_user_emb = rng.standard_normal((n_users, latent_dim)).astype(np.float32)
true_item_emb = rng.standard_normal((n_items, latent_dim)).astype(np.float32)

# train/test dict ها بر اساس Top-K
# scores = true_user_emb @ true_item_emb.T
# train_dict, test_dict = {}, {}
# for u in range(n_users):
#     ranked = scores[u].argsort()[::-1]

#     train_items = set(ranked[:20])
#     candidates  = [i for i in ranked[20:80] if i not in train_items]
#     test_items  = set(candidates[:10])

#     train_dict[u] = train_items
#     test_dict[u]  = test_items

u = 0
user_scores = scores[u]
test_items = sorted(test_dict[u])

for item in test_items:
    rank = user_scores.argsort()[::-1].tolist().index(item)
    print(f"Item {item} rank: {rank}")


# print("train_dict",train_dict)
# print("test_dict",test_dict)
# print("scores",scores)

# embedding با نویز
user_emb = (true_user_emb + noise_scale * rng.standard_normal(true_user_emb.shape)).astype(np.float32)
item_emb = (true_item_emb + noise_scale * rng.standard_normal(true_item_emb.shape)).astype(np.float32)
user_emb = np.ascontiguousarray(user_emb)
item_emb = np.ascontiguousarray(item_emb)
faiss.normalize_L2(user_emb)
faiss.normalize_L2(item_emb)

#ساخت ایندکس دقیق
dimension = latent_dim
index = faiss.IndexFlatIP(dimension)
index.add(item_emb)

# نگاشت ID به index
item_ids = np.arange(n_items)

# تعریف evaluate
def evaluate_with_faiss(
    user_embeddings: np.ndarray,
    item_embeddings: np.ndarray,
    train_dict: dict,
    test_dict: dict,
    K: int = 10,
    retrieval_k: int = 200,
    verbose: bool = True,
    item_ids: np.ndarray = None
) -> dict:
    precisions, recalls, ndcgs, hits = [], [], [], []
    valid_users = [u for u in test_dict if u in train_dict and len(test_dict[u]) > 0]

    for user_id in valid_users:
        query = user_embeddings[user_id].reshape(1, -1)
        _, raw_indices = index.search(query, retrieval_k)

        if user_id == 0:
          print("\nUser ID:", user_id)
          print("→ FAISS raw_indices:", raw_indices[0][:20])  # فقط ۲۰تای اول
          print("→ mapped item_ids:", [item_ids[i] for i in raw_indices[0][:20]])
          print("→ train items:", sorted(train_dict[user_id]))
          print("→ test items:", sorted(test_dict[user_id]))

        # نگاشت از ایندکس FAISS به item_id اصلی
        retrieved_ids = [item_ids[i] for i in raw_indices[0]] if item_ids is not None else list(raw_indices[0])

        # حذف آیتم‌های دیده‌شده
        pred = [item for item in retrieved_ids if item not in train_dict[user_id]]
        if user_id == 0:
          print("$$pred after removing train:", pred[:20])
          print("$$ground-truth test:", test_dict[user_id])
          print("$$intersection pred ∩ test:", set(pred[:10]) & test_dict[user_id])
        if len(pred) < K:
            continue

        gt = test_dict[user_id]
        precisions.append(precision_at_k(pred, gt, K))
        if user_id == 0:
          print("Precision:", precision_at_k(pred, gt, K))
          print("Recall:", recall_at_k(pred, gt, K))

        recalls.append(recall_at_k(pred, gt, K))
        ndcgs.append(ndcg_at_k(pred, gt, K))
        hits.append(hit_rate_at_k(pred, gt, K))

    if verbose:
        print(f"Total evaluated users: {len(precisions)}/{len(valid_users)}")
        if len(precisions) == 0:
            print("No users had enough valid predictions for evaluation.")
        else:
            print(f"Precision@{K}: {np.mean(precisions):.4f}")
            print(f"Recall@{K}: {np.mean(recalls):.4f}")
            print(f"NDCG@{K}: {np.mean(ndcgs):.4f}")
            print(f"HitRate@{K}: {np.mean(hits):.4f}")

    return {
        "precision": np.mean(precisions) if precisions else 0.0,
        "recall": np.mean(recalls) if recalls else 0.0,
        "ndcg": np.mean(ndcgs) if ndcgs else 0.0,
        "hit_rate": np.mean(hits) if hits else 0.0,
        "evaluated_users": len(precisions)
    }

# توابع ارزیابی
def precision_at_k(pred, gt, k):
    return len(set(pred[:k]) & gt) / k

def recall_at_k(pred, gt, k):
    return len(set(pred[:k]) & gt) / len(gt)

def ndcg_at_k(pred, gt, k):
    pred_k = pred[:k]
    dcg = sum([1 / np.log2(i + 2) if pred_k[i] in gt else 0 for i in range(len(pred_k))])
    idcg = sum([1 / np.log2(i + 2) for i in range(min(len(gt), k))])
    return dcg / idcg if idcg > 0 else 0

def hit_rate_at_k(pred, gt, k):
    return 1.0 if len(set(pred[:k]) & gt) > 0 else 0.0

#   اجرا
metrics = evaluate_with_faiss(
    user_embeddings=user_emb,
    item_embeddings=item_emb,
    train_dict=train_dict,
    test_dict=test_dict,
    K=10,
    retrieval_k=80,      # چون test از 20:80 ساخته شده
    verbose=True,
    item_ids=item_ids
)



🔍 User ID: 0
→ FAISS raw_indices: [409 861 853 343 458 634 771 981 460 174 972 393 152 425 617 143 696 515
 121 860]
→ mapped item_ids: [np.int64(409), np.int64(861), np.int64(853), np.int64(343), np.int64(458), np.int64(634), np.int64(771), np.int64(981), np.int64(460), np.int64(174), np.int64(972), np.int64(393), np.int64(152), np.int64(425), np.int64(617), np.int64(143), np.int64(696), np.int64(515), np.int64(121), np.int64(860)]
→ train items: [np.int64(152), np.int64(174), np.int64(298), np.int64(343), np.int64(393), np.int64(409), np.int64(425), np.int64(435), np.int64(458), np.int64(460), np.int64(515), np.int64(617), np.int64(634), np.int64(696), np.int64(771), np.int64(853), np.int64(860), np.int64(861), np.int64(972), np.int64(981)]
→ test items: [np.int64(121), np.int64(143), np.int64(172), np.int64(288), np.int64(314), np.int64(358), np.int64(613), np.int64(826), np.int64(970), np.int64(987)]
→ pred after removing train: [np.int64(143), np.int64(121), np.int64(172), np.int